In [ ]:
import numpy as np
import cv2
import math

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_eye_tree_eyeglasses.xml')
detector_params = cv2.SimpleBlobDetector_Params()
detector_params.filterByArea = True
detector_params.maxArea = 1500
detector = cv2.SimpleBlobDetector_create(detector_params)
contours = []
font = cv2.FONT_HERSHEY_SIMPLEX

def detect_faces(img, cascade):
    gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    coords = cascade.detectMultiScale(gray_frame, 1.3, 5)
    if len(coords) > 1:
        biggest = (0, 0, 0, 0)
        for i in coords:
            if i[3] > biggest[3]:
                biggest = i
        biggest = np.array([i], np.int32)
    elif len(coords) == 1:
        biggest = coords
    else:
        return None
    for (x, y, w, h) in biggest:
        frame = img[y:y + h, x:x + w]
    return frame

def detect_eyes(img, cascade):
    gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    eyes = cascade.detectMultiScale(gray_frame, 1.3, 5)  # detect eyes
    width = np.size(img, 1)  # get face frame width
    height = np.size(img, 0)  # get face frame height
    left_eye = None
    right_eye = None
    for (x, y, w, h) in eyes:
        if y > height / 2:
            pass
        eyecenter = x + w / 2  # get the eye center
        if eyecenter < width * 0.5:
            left_eye = img[y:y + h, x:x + w]
        else:
            right_eye = img[y:y + h, x:x + w]
    return left_eye, right_eye


def cut_eyebrows(img):
    height, width = img.shape[:2]
    eyebrow_h = int(height / 4)
    img = img[eyebrow_h:height, 0:width]  # cut eyebrows out (15 px)

    return img


def blob_process(img, threshold, detector):
    gray_frame = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, img = cv2.threshold(gray_frame, threshold, 255, cv2.THRESH_BINARY)
    img = cv2.erode(img, None, iterations=2)
    img = cv2.dilate(img, None, iterations=4)
    img = cv2.medianBlur(img, 5)
    keypoints = detector.detect(img)
    contours, hierarchy = cv2.findContours(img, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
    print (contours)
    print(keypoints)
    return keypoints, contours


def nothing(x):
    pass

def main():
    cap = cv2.VideoCapture(0)
    cv2.namedWindow('image')
    cv2.createTrackbar('threshold', 'image', 45, 255, nothing)
    threshold = r = cv2.getTrackbarPos('threshold', 'image')

    while True:
        _, frame = cap.read()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        for (x, y, w, h) in faces:
            cv2.rectangle (frame, (x, y), (x+w, y+h), (255,0,0), 2)
            roi_gray =gray[y:y+w, x:x+w]
            roi_color= frame[y:y+h, x:x+w]
            eyes= eye_cascade.detectMultiScale(roi_gray, 1.3, 5)
            for (ex, ey, ew, eh) in eyes:
                cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0,255,0), 2)
        face_frame = detect_faces(frame, face_cascade)
        if face_frame is not None:
            eyes = detect_eyes(face_frame, eye_cascade)
            for left_eye in eyes:
                if left_eye is not None:          
                    left_eye = cut_eyebrows(left_eye)
                    keypoints , contours = blob_process(left_eye, threshold, detector)                                     
                    left_eye = cv2.drawKeypoints(left_eye, keypoints, left_eye, (0, 0, 255), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
                    cv2.putText(frame, "Left Iris Detected", (10, 30),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                    #cv2.drawContours(left_eye, contours, -1, (0, 255, 0), 2, cv2.LINE_AA)

                    for contour in contours:
                        area = cv2.contourArea(contour)
                        rect = cv2.boundingRect(contour)
                        x, y, w, h = rect
                        radius = 0.15 * (w + h)
                        area_calc = (math.pi * math.pow(radius, 2.0))

                    if (abs((float(h)/float(w))) <= 0.775):
                        if ((abs(float(area)/float(area_calc)))<1):
                            text1='Ptosis confirmed in Left Eye'
                            cv2.putText(frame, text1, (10, 50),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                            break
                        else:
                            text1='Ptosis suspected in Left Eye'
                            cv2.putText(frame, text1, (10, 50),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                            break      
                    else:
                        text1='No Ptosis suspected in Left Eye'
                        cv2.putText(frame, text1, (10, 50),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                        break          
                        
                                
            for right_eye in eyes:
                if right_eye is not None:
                    eye = cut_eyebrows(right_eye)
                    keypoints , contours = blob_process(right_eye, threshold, detector)                                     
                    right_eye = cv2.drawKeypoints(right_eye, keypoints, right_eye, (0, 0, 255), cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
                    cv2.putText(frame, "Right Iris Detected", (10, 70),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                    #cv2.drawContours(right_eye, contours, -1, (0, 255, 0), 2, cv2.LINE_AA)

                    for contour in contours:
                        area = cv2.contourArea(contour)
                        rect = cv2.boundingRect(contour)
                        x, y, w, h = rect
                        radius = 0.15 * (w + h)
                        area_calc = (math.pi * math.pow(radius, 2.0))

                    if (abs((float(h)/float(w))) <= 0.775) is True:
                            if ((abs(float(area)/float(area_calc)))<1) is True:               
                                text1='Ptosis confirmed in Right Eye'
                                cv2.putText(frame, text1, (10, 90),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                                break
                            else:
                                text1='Ptosis suspected in Right Eye'
                                cv2.putText(frame, text1, (10, 90),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                                break           

                    else:
                        text1='No Ptosis suspected in Right Eye'
                        cv2.putText(frame, text1, (10, 90),font, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                        break                   
            
        
        cv2.imshow('image', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

